In [2]:
!pip install -q pinecone langchain langchain-pinecone langchain-core langchain-community langchain_google_genai pymupdf tiktoken python-dotenv langchain_huggingface sentence-transformers transformers pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 113.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 

In [1]:
from langchain_community.document_loaders import PyMuPDFLoader,DirectoryLoader,PDFPlumberLoader
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_8125/1045642509.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader,DirectoryLoader,PDFPlumberLoader


# Document Ingestion

In [2]:
# loader = DirectoryLoader(
#     path="/content/Handbook",
#     glob = "*.pdf",
#     loader_cls = PyMuPDFLoader,
#     silent_errors=True
# )

loader = PDFPlumberLoader('/content/handbook_full_removed.pdf')
docs = loader.load()
print(len(docs))

102


In [3]:
print(docs[101].page_content)

PHILOSOPHY OF LIFE
'Art of Giving' is a not-for-profit initiative for spreading, supporting and promoting the practice of giving
around the world. It is based on the philosophy of life of Prof. Achyuta Samanta, who has struggled
through an experience of poverty, hunger, humiliation in receiving and pleasure in giving from his
childhood. He gives the credit of all his success to 'Art of Giving' and has been working relentlessly to
achieve zero poverty, zero hunger and zero illiteracy since 1987.



In [4]:
docs[82].page_content

'SOT Student Handbook 2025 - 26\n15. Contact Persons for different Activities\nInformation Related to Contact Persons Mobile No\nAcademic & Curricular Activities Dean/Director General of the School\nStudent Services Prof. (Dr.) Samaresh Mishra 9437189722\nStudent Affairs & Alumni & UDC Dr. Shyam Sundar Behura 9178358687\nTraining and Placement Prof. (Dr.) Prachet Bhuyan 9337019321\nChairperson - Student Counseling Dr. Pranab Mohapatra 7752085533\nChairperson - Grievance Redressal Committee Dr. Srinivas Pattanaik 7381008280\nAsst. Director - International Relation Office (IRO) Dr. Samuel Rout 7735389456\nDirector - General Administration & Hostel Affairs Mr. Snehasish Rout 9337107807\nWarden - Boys’ Hostel Dr. Prasanta Kumar Patra 9861103036\nWarden - Girls’ Hostel Dr. Kajal Parashar 9438730874\nDean - KIIT Student Activity Centre (KSAC) Dr. Krishna Chakraborty 8658115387\nDy. Director – NCC Mr. Tribikram Mohanty 9437230562\nProgramme Coordinator – NSS Dr. Prabhat Kumar Rout 9040089310\

In [5]:
print(docs[10].metadata)

{'source': '/content/handbook_full_removed.pdf', 'file_path': '/content/handbook_full_removed.pdf', 'page': 10, 'total_pages': 102, 'Producer': 'iLovePDF', 'ModDate': 'D:20260608140138Z'}


Preprocessing the text for document

In [6]:
import re
def clean_text(text):

    text = re.sub(r'KIIT Publication Cell.*', '', text)

    text = re.sub(r'SOT Student Handbook.*', '', text)

    text = re.sub(r'Page\s+\d+', '', text)

    # text = re.sub(r'\b\d{10}\b','',text)

    text = re.sub(r'\n+', '\n', text)

    return text.strip()

In [7]:
for doc in docs:
    doc.page_content = clean_text(doc.page_content)

In [8]:
docs[82].page_content

'15. Contact Persons for different Activities\nInformation Related to Contact Persons Mobile No\nAcademic & Curricular Activities Dean/Director General of the School\nStudent Services Prof. (Dr.) Samaresh Mishra 9437189722\nStudent Affairs & Alumni & UDC Dr. Shyam Sundar Behura 9178358687\nTraining and Placement Prof. (Dr.) Prachet Bhuyan 9337019321\nChairperson - Student Counseling Dr. Pranab Mohapatra 7752085533\nChairperson - Grievance Redressal Committee Dr. Srinivas Pattanaik 7381008280\nAsst. Director - International Relation Office (IRO) Dr. Samuel Rout 7735389456\nDirector - General Administration & Hostel Affairs Mr. Snehasish Rout 9337107807\nWarden - Boys’ Hostel Dr. Prasanta Kumar Patra 9861103036\nWarden - Girls’ Hostel Dr. Kajal Parashar 9438730874\nDean - KIIT Student Activity Centre (KSAC) Dr. Krishna Chakraborty 8658115387\nDy. Director – NCC Mr. Tribikram Mohanty 9437230562\nProgramme Coordinator – NSS Dr. Prabhat Kumar Rout 9040089310\nCoordinator -Youth Red Cross (Y

# Chunking

In [9]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1300,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        ". ",
        "•",
        ":"
    ]
)

ndocs = splitter.split_documents(docs)

print(ndocs[82].page_content)
print(len(ndocs))


activities with an understanding of the limitations.
f) The engineer and society: Ability to apply reasoning informed by the contextual knowledge to
assess societal, health, safety, legal and cultural issues and the consequent responsibilities
relevant to the professional engineering practice.
g) Environment and sustainability: Ability to understand the impact of the professional
engineering solutions in societal and environmental contexts, and demonstrate the knowledge
of, and need for sustainable development.
h) Ethics: Ability to apply ethical principles and commit to professional ethics and responsibilities
and norms of the engineering practice.
i) Individual and team: Ability to function effectively as an individual, and as a member or leader
in diverse teams, and in multidisciplinary settings.
30
244


# Embedding the chunks

In [10]:
embedding_model = HuggingFaceEmbeddings(
    model_name="intfloat/e5-large-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/67.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [ ]:
from pinecone import Pinecone,ServerlessSpec
pc = Pinecone(api_key='')
index_name = "bandhu-db"

if pc.has_index(index_name):
    pc.delete_index(index_name)

pc.create_index(
    name = index_name,
    dimension = 1024,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

{
    "name": "bandhu-db",
    "metric": "cosine",
    "host": "bandhu-db-7uweph7.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1024,
    "deletion_protection": "disabled",
    "tags": null
}

In [13]:
index_name = "bandhu-db"
vector_store = PineconeVectorStore(
    index_name = index_name,
    embedding= embedding_model
)

In [ ]:
# index_name = "bandhu-db"
# pinecone_index = vector_store.index
# result = pinecone_index.fetch(
#     ids=[ '46ec788b-2193-4dd3-b8c4-3e9f5a56d9a3']
# )
# result

In [18]:
query = "phone number of Contact Persons for different Activities"

results = vector_store.similarity_search_with_score(
    query,
    k=3
)

for doc, score in results:
    print(score)
    print(doc.page_content[:1000])
    print("="*50)

0.849298537
15. Contact Persons for different Activities
Information Related to Contact Persons Mobile No
Academic & Curricular Activities Dean/Director General of the School
Student Services Prof. (Dr.) Samaresh Mishra 9437189722
Student Affairs & Alumni & UDC Dr. Shyam Sundar Behura 9178358687
Training and Placement Prof. (Dr.) Prachet Bhuyan 9337019321
Chairperson - Student Counseling Dr. Pranab Mohapatra 7752085533
Chairperson - Grievance Redressal Committee Dr. Srinivas Pattanaik 7381008280
Asst. Director - International Relation Office (IRO) Dr. Samuel Rout 7735389456
Director - General Administration & Hostel Affairs Mr. Snehasish Rout 9337107807
Warden - Boys’ Hostel Dr. Prasanta Kumar Patra 9861103036
Warden - Girls’ Hostel Dr. Kajal Parashar 9438730874
Dean - KIIT Student Activity Centre (KSAC) Dr. Krishna Chakraborty 8658115387
Dy. Director – NCC Mr. Tribikram Mohanty 9437230562
Programme Coordinator – NSS Dr. Prabhat Kumar Rout 9040089310
Coordinator -Youth Red Cross (YRC) 

# Retriever

In [20]:
retriever = vector_store.as_retriever(
    search_type = 'mmr',
    search_kwargs = {'k':3,
                     "fetch_k":25,
                     'lambda_mult':0.75}
)

In [21]:
query = "can you tell me about alcohol use in kiit?"
docs = retriever.invoke(query)
print(docs)

[Document(metadata={'ModDate': 'D:20260608140138Z', 'Producer': 'iLovePDF', 'file_path': '/content/handbook_full_removed.pdf', 'page': 49.0, 'source': '/content/handbook_full_removed.pdf', 'total_pages': 102.0}, page_content='4.11.3 Ragging:\n4.12 .Policy on Substance Abuse\n4.12.1 Objective\nTo prevent substance abuse and to create a secure, conducive atmosphere for learning\namong the students on the campus, KIIT Deemed University adheres to the following\nguidelines concerning the possession, use and/or distribution of substances of abuse:\nCannabis, Heroin, Benzodiazepines, barbiturates, Flunitrazepam, Cocaine, Ketamine,\nPsilocybin, Lysergic acid diethylamide, Amphetamine, Methamphetamines, MDMA,\nPhencyclidine, GHB, Methaqualone, Inhalants and any other drugs and substances mentioned in\nThe Narcotic Drugs and Psychotropic Act 1985.\n47'), Document(metadata={'ModDate': 'D:20260608140138Z', 'Producer': 'iLovePDF', 'file_path': '/content/handbook_full_removed.pdf', 'page': 61.0, 's

In [22]:
context = "";
for doc in docs:
  context += doc.page_content

print(context)

4.11.3 Ragging:
4.12 .Policy on Substance Abuse
4.12.1 Objective
To prevent substance abuse and to create a secure, conducive atmosphere for learning
among the students on the campus, KIIT Deemed University adheres to the following
guidelines concerning the possession, use and/or distribution of substances of abuse:
Cannabis, Heroin, Benzodiazepines, barbiturates, Flunitrazepam, Cocaine, Ketamine,
Psilocybin, Lysergic acid diethylamide, Amphetamine, Methamphetamines, MDMA,
Phencyclidine, GHB, Methaqualone, Inhalants and any other drugs and substances mentioned in
The Narcotic Drugs and Psychotropic Act 1985.
47Abuse of library books, periodicals, equipment, furniture, and different facilities offered to the
readers like tearing out pages from a book, putting marks on different pages, underlining, removing
bar-codes, removing labels and electronic theft devices,damaging or defacing library books or any
materials is strictly prohibited.
All users of the library are advised to keep their 

#Langchain Chains -> Augmenatation and generation

In [23]:
from langchain_core.runnables import RunnableSequence

In [24]:
template = PromptTemplate(
    template = """You are an expert Student Handbook Guidelines Assistant.
Your role is to answer student questions using ONLY the information provided in the retrieved handbook context.
Instructions
Carefully read:
The retrieved handbook context
The student's question
Provide a clear, accurate, and student-friendly response based strictly on the handbook rules.
Use simple, direct language. Explain policies in an easy-to-understand way without using complex legal or administrative wording.
Remain fully grounded in the retrieved context:
Do NOT invent policies, penalties, procedures, deadlines, or exceptions.
Do NOT assume information that is not explicitly stated.
Do NOT use outside knowledge.
If faculty names, contact numbers, office details,
or tabular information are present in the retrieved
context, extract them exactly as written.

Do not summarize numbers.
Do not alter phone numbers.
Do not infer missing digits.
If the answer is partially available:
Answer only the portion supported by the context.
Clearly mention what is not specified in the handbook context.
If the retrieved context does not contain enough information:
Say that the handbook does not provide a clear answer.
Suggest checking with the relevant university/college authority or handbook section.
Do NOT fabricate an answer.
Maintain a helpful, professional, and neutral tone.
When appropriate, structure the response as:
Answer concisely (50-150 words).
Only expand if the handbook explicitly provides detailed procedures.
Relevant Rule / Guideline
STRICT RULES:
1. Answer ONLY from the most directly relevant lines.
2. Ignore unrelated handbook text even if retrieved.
3. Do not broaden to nearby policies or committees unless explicitly asked.
4. Every factual statement must be directly supported by retrieved text.
Retrieved Context
{context}
Student Question
{question} """,
    input_variables=['context','question']
)

In [ ]:
query = "who is the contact person for youth red cross at kiit and how can i contact them ?"
docs = retriever.invoke(query)
context = ""
for doc in docs:
  context += doc.page_content
prompt = template.invoke({'context':context,'question':query})

In [ ]:
query = "who are the point of contacts for the sports facilities?"
docs = retriever.invoke(query)
context = ""
for doc in docs:
  context += doc.page_content
prompt = template.invoke({'context':context,'question':query})

In [39]:
result = model.invoke(prompt)
print(result.text)

Answer:

According to the retrieved handbook context, the points of contact for the sports facilities are not explicitly stated. However, it is mentioned that "players from across the country every year" can register for the sports complex and that students can receive notice for selection trials of University Team to participate in various Inter University competitions.

It is unclear who the point of contact would be for the sports facilities. I would recommend checking with the relevant university/college authority or handbook section for more information.

Relevant guideline: Since the retrieved context does not provide a clear answer, I am unable to provide a detailed response. I will refrain from fabricating an answer and instead suggest checking with the relevant authority.
